In [13]:
original_poem="""One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2" # Using gpt2 as a common causal language model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [15]:
def get_replacement_word(tokenizer, model, sentence_prefix, word_index):
    """
    Generates the next word prediction for the given prefix using a transformers model
    and returns the token at the specified index from the sorted vocabulary.
    """
    try:
        # Encode the sentence prefix
        input_ids = tokenizer.encode(sentence_prefix, return_tensors='pt')

        # Get the model's predictions for the next token
        with torch.no_grad():
            outputs = model(input_ids)
            predictions = outputs.logits

        # Get the logits for the last token in the sequence
        next_token_logits = predictions[0, -1, :]

        # Sort the logits to find the most likely tokens
        sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)

        # Get the token ID at the specified word_index
        if word_index < len(sorted_indices):
            next_token_id = sorted_indices[word_index].item()
            # Decode the token ID back to a word
            return tokenizer.decode(next_token_id).strip()
        else:
            # If word_index is out of bounds, return the most likely word
            next_token_id = sorted_indices[0].item()
            return tokenizer.decode(next_token_id).strip()
    except Exception as e:
        print(f"Error generating content: {e}")
        return "[error]" # Placeholder for error cases

In [16]:
import re # Import the regular expression module

# Define the index for the replacement word (7th most likely, so index 6)
replacement_word_index = 23

modified_lines = []
for line in original_poem.split('\n'):
    words = line.split()
    if not words:
        modified_lines.append(line) # Keep empty lines as is
        continue

    # Extract the original last word with its punctuation
    original_last_word_full = words[-1]

    # Separate the word part from the trailing punctuation
    # This regex captures the word characters (alphanumeric) and then any trailing punctuation
    # The single quote has been escaped to prevent SyntaxError
    match = re.match(r'(\w*)([.,;!?"\"]*)$', original_last_word_full)

    trailing_punctuation = ''
    if match:
        # We don't need the 'original_last_word_text' part for the prefix, just the punctuation
        trailing_punctuation = match.group(2)

    # Form the sentence prefix for the model (all words except the last one)
    sentence_prefix_for_model = ' '.join(words[:-1])

    # Get the replacement word from the model (this will be a clean word without punctuation)
    new_last_word = get_replacement_word(tokenizer, model, sentence_prefix_for_model, replacement_word_index)

    # Reconstruct the new last word with the original punctuation
    new_last_word_with_punct = new_last_word + trailing_punctuation

    # Reconstruct the modified line
    # Handle cases where the sentence_prefix_for_model might be empty (e.g., a line with only one word)
    if sentence_prefix_for_model:
        modified_line = f"{sentence_prefix_for_model} {new_last_word_with_punct}"
    else:
        modified_line = new_last_word_with_punct

    modified_lines.append(modified_line)

modified_poem = '\n'.join(modified_lines)

print("Original Poem:\n")
print(original_poem)
print("\n" + "-" * 30 + "\n")
print(f"Modified Poem (using {replacement_word_index+1}th replacement):")
print(modified_poem)

Original Poem:

One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.

------------------------------

Modified Poem (using 24th replacement):
One must have a mind of thy
To regard the frost and the loss
Of the pine-trees crusted with sugar;
And have been cold a long times
To behold the junipers shagged with venom,
The spruces rough in the distant forests
Of the January sun; and not to this
Of any misery in the sound of the news,
In the sound of a few cheers,
Which is the sound of the explosion
Full of the sam